- PTC 不是“Claude 多调几次工具”，而是“Claude 先拿到一个可执行代码的沙箱，再在这段代码里程序化地调用别的工具”。外层对话只保留必要结论，中间的检索、过滤、重试、聚合都留在沙箱里做。
    - 代码里程序化调工具
- 传统 tool use：模型一轮一轮地决定 先调 A，再看结果，再调 B。
  - PTC：模型先写一个“小程序”，后续 A/B/C 的具体调用顺序由程序在运行时决定。
  - plan of code execution

```python
def run():
    hits = tool("search_docs", {"q": "ptc architecture"})
    filtered = [x for x in hits if "sandbox" in x["text"].lower()]
    return {
      "summary": f"found {len(filtered)} relevant docs",
      "top_ids": [x["id"] for x in filtered[:5]]
    }
```

- 不是 agent 一次性把“所有工具调用结果”都想好，而是 agent 一次性输出“编排这些工具的代码/程序”。
1. 外层 Claude 先产出一次 code_execution 调用。
2. 这个调用里带着一段 Python 代码。
3. 代码在 sandbox 里运行，运行到 tool(...) 时，再去实际调用 server tools。
4. 工具结果返回给这段代码。
5. 代码根据结果继续分支、循环、过滤、重试、聚合。
6. 最后把压缩后的最终结果返回给外层模型。